In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import MinMaxScaler

# -------------------------------
# CONFIG
# -------------------------------
DATA_PATH = r"C:\Users\mayil\Downloads\jan to may police violation_anonymized791b166.csv"
MODEL_DIR = "model/"
EARTH_RADIUS_KM = 6371

# -------------------------------
# LOAD DATA
# -------------------------------
df = pd.read_csv(DATA_PATH)

df["created_datetime"] = pd.to_datetime(
    df["created_datetime"],
    utc=True,
    format="mixed",
    errors="coerce"
)
df["hour"] = df["created_datetime"].dt.hour

# -------------------------------
# SPATIAL FEATURES
# -------------------------------
coords = df[["latitude", "longitude"]].to_numpy()

epsilon = 0.08 / EARTH_RADIUS_KM  # ~80 meters

dbscan = DBSCAN(
    eps=epsilon,
    min_samples=8,
    algorithm="ball_tree",
    metric="haversine"
)

# -------------------------------
# TRAIN CLUSTERING MODEL
# -------------------------------
dbscan.fit(np.radians(coords))

df["cluster"] = dbscan.labels_
df = df[df["cluster"] != -1]

# -------------------------------
# FEATURE ENGINEERING
# -------------------------------
vehicle_weight = {
    "SCOOTER": 0.3,
    "CAR": 0.6,
    "MAXI-CAB": 0.9,
    "LCV": 0.8
}

df["vehicle_weight"] = df["vehicle_type"].map(vehicle_weight).fillna(0.5)

def violation_score(v):
    if "MAIN ROAD" in v or "ROAD CROSSING" in v:
        return 1.0
    if "NO PARKING" in v:
        return 0.7
    return 0.5

df["violation_severity"] = df["violation_type"].astype(str).apply(violation_score)

# -------------------------------
# AGGREGATE HOTSPOTS
# -------------------------------
hotspots = []

for cid, g in df.groupby("cluster"):
    raw_score = (
        0.45 * len(g) +
        0.30 * g["vehicle_weight"].mean() * 100 +
        0.25 * g["violation_severity"].mean() * 100
    )

    hotspots.append({"cluster": cid, "raw_score": raw_score})

hotspots_df = pd.DataFrame(hotspots)

# -------------------------------
# TRAIN SCALER
# -------------------------------
scaler = MinMaxScaler(feature_range=(20, 95))
hotspots_df["score"] = scaler.fit_transform(
    hotspots_df[["raw_score"]]
)

# -------------------------------
# SAVE MODELS
# -------------------------------
joblib.dump(dbscan, MODEL_DIR + "dbscan.pkl")
joblib.dump(scaler, MODEL_DIR + "scaler.pkl")

print("✅ Training completed")
print("✅ DBSCAN and Scaler saved")

✅ Training completed
✅ DBSCAN and Scaler saved
